In [1]:
from collections import Counter
import pandas as pd

In [2]:
texts = [
    "I love machine learning.",
    "I love deep learning.",
    "我喜欢机器学习",
    "我喜欢深度学习",
    "机器学习可以帮助计算机从数据中学习"
]

texts

['I love machine learning.',
 'I love deep learning.',
 '我喜欢机器学习',
 '我喜欢深度学习',
 '机器学习可以帮助计算机从数据中学习']

In [3]:
def whitespace_tokenizer(text):
    """
    最简单的英文 tokenizer：按空格切分
    """
    return text.split()


english_text = "I love machine learning"

tokens = whitespace_tokenizer(english_text)

print("原始文本：", english_text)
print("切分结果：", tokens)

原始文本： I love machine learning
切分结果： ['I', 'love', 'machine', 'learning']


In [4]:
english_text_with_punc = "I love machine learning."

tokens = whitespace_tokenizer(english_text_with_punc)

print("原始文本：", english_text_with_punc)
print("切分结果：", tokens)

原始文本： I love machine learning.
切分结果： ['I', 'love', 'machine', 'learning.']


In [5]:
def char_tokenizer(text):
    """
    中文字符级 tokenizer：一个字符一个 token
    """
    return list(text)


chinese_text = "我喜欢机器学习"

tokens = char_tokenizer(chinese_text)

print("原始文本：", chinese_text)
print("切分结果：", tokens)

原始文本： 我喜欢机器学习
切分结果： ['我', '喜', '欢', '机', '器', '学', '习']


In [6]:
word_dict = [
    "我",
    "喜欢",
    "机器学习",
    "深度学习",
    "可以",
    "帮助",
    "计算机",
    "从",
    "数据",
    "中",
    "学习"
]

word_dict = sorted(word_dict, key=len, reverse=True)

word_dict

['机器学习', '深度学习', '计算机', '喜欢', '可以', '帮助', '数据', '学习', '我', '从', '中']

In [7]:
def longest_match_tokenizer(text, word_dict):
    """
    简化版中文最长匹配 tokenizer
    
    思路：
    从当前位置开始，优先匹配词典里最长的词。
    如果匹配不到，就把当前字符作为一个 token。
    """
    tokens = []
    i = 0
    
    while i < len(text):
        matched = False
        
        for word in word_dict:
            if text[i:i + len(word)] == word:
                tokens.append(word)
                i += len(word)
                matched = True
                break
        
        if not matched:
            tokens.append(text[i])
            i += 1
    
    return tokens


chinese_text = "我喜欢机器学习"

tokens = longest_match_tokenizer(chinese_text, word_dict)

print("原始文本：", chinese_text)
print("切分结果：", tokens)

原始文本： 我喜欢机器学习
切分结果： ['我', '喜欢', '机器学习']


In [8]:
text = "机器学习可以帮助计算机从数据中学习"

char_tokens = char_tokenizer(text)
word_tokens = longest_match_tokenizer(text, word_dict)

print("原始文本：")
print(text)

print("\n字符级切分：")
print(char_tokens)

print("\n词级切分：")
print(word_tokens)

print("\n字符级 token 数量：", len(char_tokens))
print("词级 token 数量：", len(word_tokens))

原始文本：
机器学习可以帮助计算机从数据中学习

字符级切分：
['机', '器', '学', '习', '可', '以', '帮', '助', '计', '算', '机', '从', '数', '据', '中', '学', '习']

词级切分：
['机器学习', '可以', '帮助', '计算机', '从', '数据', '中', '学习']

字符级 token 数量： 17
词级 token 数量： 8


In [9]:
def mixed_tokenizer(text):
    """
    简化混合 tokenizer：
    - 英文句子：按空格切分，并简单处理句号
    - 中文句子：使用最长匹配分词
    """
    if any("\u4e00" <= ch <= "\u9fff" for ch in text):
        return longest_match_tokenizer(text, word_dict)
    else:
        text = text.replace(".", " .")
        return whitespace_tokenizer(text)


tokenized_texts = [mixed_tokenizer(text) for text in texts]

for text, tokens in zip(texts, tokenized_texts):
    print("原始文本：", text)
    print("tokens：", tokens)
    print("-" * 50)

原始文本： I love machine learning.
tokens： ['I', 'love', 'machine', 'learning', '.']
--------------------------------------------------
原始文本： I love deep learning.
tokens： ['I', 'love', 'deep', 'learning', '.']
--------------------------------------------------
原始文本： 我喜欢机器学习
tokens： ['我', '喜欢', '机器学习']
--------------------------------------------------
原始文本： 我喜欢深度学习
tokens： ['我', '喜欢', '深度学习']
--------------------------------------------------
原始文本： 机器学习可以帮助计算机从数据中学习
tokens： ['机器学习', '可以', '帮助', '计算机', '从', '数据', '中', '学习']
--------------------------------------------------


In [10]:
special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]

all_tokens = []

for tokens in tokenized_texts:
    all_tokens.extend(tokens)

token_counts = Counter(all_tokens)

vocab = special_tokens + sorted(token_counts.keys())

token2id = {token: idx for idx, token in enumerate(vocab)}
id2token = {idx: token for token, idx in token2id.items()}

print("词表大小：", len(vocab))
print("词表：")
print(vocab)

词表大小： 22
词表：
['[PAD]', '[UNK]', '[BOS]', '[EOS]', '[MASK]', '.', 'I', 'deep', 'learning', 'love', 'machine', '中', '从', '可以', '喜欢', '学习', '帮助', '我', '数据', '机器学习', '深度学习', '计算机']


In [11]:
pad_id = token2id["[PAD]"]
unk_id = token2id["[UNK]"]
bos_id = token2id["[BOS]"]
eos_id = token2id["[EOS]"]
mask_id = token2id["[MASK]"]


def encode(text, add_special_tokens=True):
    """
    把文本转成 token id
    """
    tokens = mixed_tokenizer(text)
    
    if add_special_tokens:
        tokens = ["[BOS]"] + tokens + ["[EOS]"]
    
    ids = [token2id.get(token, unk_id) for token in tokens]
    
    return ids


def decode(ids, remove_special_tokens=False):
    """
    把 token id 转回 token
    """
    tokens = [id2token.get(idx, "[UNK]") for idx in ids]
    
    if remove_special_tokens:
        tokens = [
            token for token in tokens
            if token not in ["[PAD]", "[BOS]", "[EOS]", "[MASK]"]
        ]
    
    return tokens

In [12]:
text = "我喜欢机器学习"

ids = encode(text)
tokens = decode(ids)

print("原始文本：", text)
print("编码 ids：", ids)
print("解码 tokens：", tokens)

原始文本： 我喜欢机器学习
编码 ids： [2, 17, 14, 19, 3]
解码 tokens： ['[BOS]', '我', '喜欢', '机器学习', '[EOS]']


In [13]:
def batch_encode(texts, max_length=10, padding=True, truncation=True):
    """
    批量编码文本，输出 input_ids 和 attention_mask
    """
    batch_input_ids = []
    batch_attention_mask = []
    
    for text in texts:
        ids = encode(text)
        
        if truncation:
            ids = ids[:max_length]
        
        attention_mask = [1] * len(ids)
        
        if padding:
            pad_len = max_length - len(ids)
            
            if pad_len > 0:
                ids = ids + [pad_id] * pad_len
                attention_mask = attention_mask + [0] * pad_len
        
        batch_input_ids.append(ids)
        batch_attention_mask.append(attention_mask)
    
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask
    }

In [14]:
batch = batch_encode(texts, max_length=10)

batch

{'input_ids': [[2, 6, 9, 10, 8, 5, 3, 0, 0, 0],
  [2, 6, 9, 7, 8, 5, 3, 0, 0, 0],
  [2, 17, 14, 19, 3, 0, 0, 0, 0, 0],
  [2, 17, 14, 20, 3, 0, 0, 0, 0, 0],
  [2, 19, 13, 16, 21, 12, 18, 11, 15, 3]],
 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
  [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
  [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
  [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [15]:
rows = []

for text, input_ids, attention_mask in zip(
    texts,
    batch["input_ids"],
    batch["attention_mask"]
):
    rows.append({
        "原始文本": text,
        "tokens": decode(input_ids),
        "input_ids": input_ids,
        "attention_mask": attention_mask
    })

df = pd.DataFrame(rows)

df

,原始文本,tokens,input_ids,attention_mask
0,I love machine learning.,"[[BOS], I, love, machine, learning, ., [EOS], ...","[2, 6, 9, 10, 8, 5, 3, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 0, 0, 0]"
1,I love deep learning.,"[[BOS], I, love, deep, learning, ., [EOS], [PA...","[2, 6, 9, 7, 8, 5, 3, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 0, 0, 0]"
2,我喜欢机器学习,"[[BOS], 我, 喜欢, 机器学习, [EOS], [PAD], [PAD], [PAD...","[2, 17, 14, 19, 3, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]"
3,我喜欢深度学习,"[[BOS], 我, 喜欢, 深度学习, [EOS], [PAD], [PAD], [PAD...","[2, 17, 14, 20, 3, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]"
4,机器学习可以帮助计算机从数据中学习,"[[BOS], 机器学习, 可以, 帮助, 计算机, 从, 数据, 中, 学习, [EOS]]","[2, 19, 13, 16, 21, 12, 18, 11, 15, 3]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


In [16]:
text = "我喜欢机器学习"

tokens = mixed_tokenizer(text)
tokens = ["[BOS]"] + tokens + ["[EOS]"]

gpt_rows = []

for i in range(1, len(tokens)):
    prefix = tokens[:i]
    target = tokens[i]
    
    gpt_rows.append({
        "GPT 输入 prefix": " ".join(prefix),
        "预测目标 next token": target
    })

gpt_df = pd.DataFrame(gpt_rows)

gpt_df

,GPT 输入 prefix,预测目标 next token
0,[BOS],我
1,[BOS] 我,喜欢
2,[BOS] 我 喜欢,机器学习
3,[BOS] 我 喜欢 机器学习,[EOS]


In [17]:
text = "我喜欢机器学习"

tokens = mixed_tokenizer(text)
tokens = ["[BOS]"] + tokens + ["[EOS]"]

bert_rows = []

for i in range(1, len(tokens) - 1):
    masked_tokens = tokens.copy()
    target = masked_tokens[i]
    masked_tokens[i] = "[MASK]"
    
    bert_rows.append({
        "BERT 输入": " ".join(masked_tokens),
        "预测目标": target,
        "MASK 位置": i
    })

bert_df = pd.DataFrame(bert_rows)

bert_df

,BERT 输入,预测目标,MASK 位置
0,[BOS] [MASK] 喜欢 机器学习 [EOS],我,1
1,[BOS] 我 [MASK] 机器学习 [EOS],喜欢,2
2,[BOS] 我 喜欢 [MASK] [EOS],机器学习,3


In [18]:
token_count_rows = []

for text in texts:
    tokens = mixed_tokenizer(text)
    ids = encode(text)
    
    token_count_rows.append({
        "原始文本": text,
        "tokens": tokens,
        "不含特殊 token 数量": len(tokens),
        "含 [BOS]/[EOS] 后数量": len(ids)
    })

token_count_df = pd.DataFrame(token_count_rows)

token_count_df

,原始文本,tokens,不含特殊 token 数量,含 [BOS]/[EOS] 后数量
0,I love machine learning.,"[I, love, machine, learning, .]",5,7
1,I love deep learning.,"[I, love, deep, learning, .]",5,7
2,我喜欢机器学习,"[我, 喜欢, 机器学习]",3,5
3,我喜欢深度学习,"[我, 喜欢, 深度学习]",3,5
4,机器学习可以帮助计算机从数据中学习,"[机器学习, 可以, 帮助, 计算机, 从, 数据, 中, 学习]",8,10


In [19]:
bpe_words = ["low", "lower", "newest", "widest"]

bpe_vocab = Counter(bpe_words)

bpe_vocab

Counter({'low': 1, 'lower': 1, 'newest': 1, 'widest': 1})

In [20]:
def word_to_symbols(word):
    """
    把一个单词拆成字符，并加上词尾标记
    """
    return tuple(list(word) + ["</w>"])


symbol_vocab = {
    word_to_symbols(word): freq
    for word, freq in bpe_vocab.items()
}

symbol_vocab

{('l', 'o', 'w', '</w>'): 1,
 ('l', 'o', 'w', 'e', 'r', '</w>'): 1,
 ('n', 'e', 'w', 'e', 's', 't', '</w>'): 1,
 ('w', 'i', 'd', 'e', 's', 't', '</w>'): 1}

In [21]:
def get_pair_counts(symbol_vocab):
    """
    统计所有相邻符号对出现的次数
    """
    pair_counts = Counter()
    
    for symbols, freq in symbol_vocab.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_counts[pair] += freq
    
    return pair_counts


pair_counts = get_pair_counts(symbol_vocab)

pair_counts

Counter({('l', 'o'): 2,
         ('o', 'w'): 2,
         ('w', 'e'): 2,
         ('e', 's'): 2,
         ('s', 't'): 2,
         ('t', '</w>'): 2,
         ('w', '</w>'): 1,
         ('e', 'r'): 1,
         ('r', '</w>'): 1,
         ('n', 'e'): 1,
         ('e', 'w'): 1,
         ('w', 'i'): 1,
         ('i', 'd'): 1,
         ('d', 'e'): 1})

In [22]:
def merge_pair(symbols, best_pair):
    """
    把 symbols 中出现的 best_pair 合并成一个新符号
    """
    merged = []
    i = 0
    
    while i < len(symbols):
        if (
            i < len(symbols) - 1
            and symbols[i] == best_pair[0]
            and symbols[i + 1] == best_pair[1]
        ):
            merged.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    
    return tuple(merged)


def merge_symbol_vocab(symbol_vocab, best_pair):
    """
    对整个词表执行一次合并
    """
    new_symbol_vocab = {}
    
    for symbols, freq in symbol_vocab.items():
        new_symbols = merge_pair(symbols, best_pair)
        new_symbol_vocab[new_symbols] = freq
    
    return new_symbol_vocab

In [23]:
symbol_vocab = {
    word_to_symbols(word): freq
    for word, freq in bpe_vocab.items()
}

history = []

num_merges = 6

for step in range(1, num_merges + 1):
    pair_counts = get_pair_counts(symbol_vocab)
    
    if not pair_counts:
        break
    
    best_pair, best_count = pair_counts.most_common(1)[0]
    
    history.append({
        "step": step,
        "合并符号对": best_pair,
        "出现次数": best_count,
        "合并前词表": list(symbol_vocab.keys())
    })
    
    symbol_vocab = merge_symbol_vocab(symbol_vocab, best_pair)

history_df = pd.DataFrame(history)

history_df

,step,合并符号对,出现次数,合并前词表
0,1,"(l, o)",2,"[(l, o, w, </w>), (l, o, w, e, r, </w>), (n, e..."
1,2,"(lo, w)",2,"[(lo, w, </w>), (lo, w, e, r, </w>), (n, e, w,..."
2,3,"(e, s)",2,"[(low, </w>), (low, e, r, </w>), (n, e, w, e, ..."
3,4,"(es, t)",2,"[(low, </w>), (low, e, r, </w>), (n, e, w, es,..."
4,5,"(est, </w>)",2,"[(low, </w>), (low, e, r, </w>), (n, e, w, est..."
5,6,"(low, </w>)",1,"[(low, </w>), (low, e, r, </w>), (n, e, w, est..."


In [24]:
symbol_vocab

{('low</w>',): 1,
 ('low', 'e', 'r', '</w>'): 1,
 ('n', 'e', 'w', 'est</w>'): 1,
 ('w', 'i', 'd', 'est</w>'): 1}